In [3]:
import os
from googleapiclient.discovery import build
from typing import List, Dict
import time

API_KEY = os.getenv("YOUTUBE_API_KEY")
VIDEO_ID = "5MWT_doo68k"

def get_all_comments(video_id: str, max_comments: int = 2000) -> tuple[List[str], List[str]]:
    """
    Obtém todos os comentários de um vídeo do YouTube com paginação.
    
    Args:
        video_id: ID do vídeo do YouTube
        max_comments: Número máximo de comentários a coletar
        
    Returns:
        Uma tupla com (lista_de_comentários, lista_de_ids)
    """
    youtube = build("youtube", "v3", developerKey=API_KEY)
    
    comentarios = []
    comment_ids = []
    next_page_token = None
    total_processed = 0
    
    while True:
        try:
            request = youtube.commentThreads().list(
                part="snippet,replies",
                videoId=video_id,
                maxResults=100,  # Máximo por requisição
                pageToken=next_page_token,
                textFormat="plainText"
            )
            response = request.execute()
            
            for item in response.get("items", []):
                comment_text = item["snippet"]["topLevelComment"]["snippet"]["textOriginal"]
                if len(comment_text) > 60:  # Filtro para comentários mais longos
                    comentarios.append(comment_text)
                    comment_ids.append(item['snippet']['topLevelComment']['id'])
                    total_processed += 1
                    
                    if total_processed >= max_comments:
                        return comentarios, comment_ids
            
            # Verifica se há mais páginas
            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break
                
            # Respeita a quota da API
            time.sleep(1)  # 1 segundo entre requisições
            
        except Exception as e:
            print(f"Erro ao buscar comentários: {e}")
            break
    
    return comentarios, comment_ids

# Uso:
comentarios, comment_ids = get_all_comments(VIDEO_ID, max_comments=2000)
print(f"Total de comentários coletados: {len(comentarios)}")

Total de comentários coletados: 1200


In [ ]:
comentarios

['3:03 "You can clap about that all you want. Enjoy". Get \'em Sam! Haha 😎😂',
 "I respect everyone's psition on the matter, in my opinion great interview by Sam.",
 'That last world Sam describes... If those kids get power outage for 1 week, they will die.',
 'Sam Altman never really answer. Maybe I am not enought inteligent to understand what is talking about but.....I feel he never responde',
 'AI is not about ego and who gets credit for what. Its about helping humanity evolve, push boundaries, and set higher standards for future generations. Its making non ego driven creative people a million times more creative',
 "Although I don't believe he's necessarily an evil guy, I think that if him and his company really cared deeply about the safety of the development of AI, he would've taken this interview as an opportunity to actually share with the people how they've put or are putting these safety measurements in place. Instead, I feel like he gets frustrated and defensive by any questi

In [31]:
def salvar_comentarios_em_txt(comentarios: List[str], filename: str = "comentarios.txt"):
    """Salva uma lista de comentários em um arquivo de texto."""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            for comentario in comentarios:
                comentario = comentario.replace('\n','')
                f.write(comentario + '\n')
        print(f"Comentários salvos com sucesso em '{filename}'")
    except IOError as e:
        print(f"Erro ao salvar os comentários no arquivo '{filename}': {e}")

def carregar_comentarios_de_txt(filename: str = "comentarios.txt") -> List[str]:
    """Carrega os comentários de um arquivo de texto e os retorna em uma lista."""
    comentarios_carregados = []
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for linha in f:
                comentarios_carregados.append(linha.strip())
        print(f"Comentários carregados com sucesso do arquivo '{filename}'")
    except FileNotFoundError:
        print(f"Arquivo '{filename}' não encontrado.")
    except IOError as e:
        print(f"Erro ao ler o arquivo '{filename}': {e}")
    return comentarios_carregados

In [32]:
salvar_comentarios_em_txt(comentarios=comentarios)

Comentários salvos com sucesso em 'comentarios.txt'


In [33]:
comentarios[23]

"Inside Sam's head:\nMaximize shareholder value\nOutside Sam: 😎\n\nTwo diametrically opposed values are go fast - be safe and that applies to both China and the US"

In [34]:
comentarios[23].replace('\n','')

"Inside Sam's head:Maximize shareholder valueOutside Sam: 😎Two diametrically opposed values are go fast - be safe and that applies to both China and the US"